# Best GH

In [23]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from pulp import *

In [24]:
safra = 'SAFRA'
perf = 'PERF'
score = 'SCORE'
gh = 'GH'
index = 'INDEX'
n_groups = 4

In [37]:
refs_list = sorted([int(sfr) for sfr in df[safra].unique()])
refs_list

[2306, 2307, 2308]

In [38]:
groups_list = sorted([int(grp) for grp in np.arange(1,n_groups+1)])
groups_list

[1, 2, 3, 4]

In [39]:
#idx_list = [str(i) for i in range(0,np.max([len(df[df[safra]==int(ref)]) for ref in refs_list]))]
idx_list = [[i+1 for i in range(0,len(df[df[safra]==ref]))] for ref in refs_list]
idx_list

[[1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40],
 [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40],
 [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38]]

In [40]:
idx_dict = {ref:[i for i in range(1,len(df[df[safra]==ref]) + 1)] for ref in refs_list}
idx_dict

{2306: [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40],
 2307: [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40],
 2308: [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38]}

In [ ]:
# VI1G1R1 ==> assigning vars

# BADG1R1 => (VI1G1R1*PERF[I1,R1] + VI2G1R1*PERF[I2,R1] + VI3G1R1*PERF[I3,R1] ...) / (VI1G1R1 + VI2G1R1 + VI3G1R1 + ...)
# BADG2R1 => (VI1G2R1*PERF[I1,R1] + VI2G2R1*PERF[I2,R1] + VI3G2R1*PERF[I3,R1] ...) / (VI1G2R1 + VI2G2R1 + VI3G2R1 + ...)
# BADG3R1 => (VI1G3R1*PERF[I1,R1] + VI2G3R1*PERF[I2,R1] + VI3G3R1*PERF[I3,R1] ...) / (VI1G3R1 + VI2G3R1 + VI3G3R1 + ...)
# BADG4R1 => (VI1G4R1*PERF[I1,R1] + VI2G4R1*PERF[I2,R1] + VI3G4R1*PERF[I3,R1] ...) / (VI1G4R1 + VI2G4R1 + VI3G4R1 + ...)

# BADG1R2 => (VI1G1R2*PERF[I1,R2] + VI2G1R2*PERF[I2,R2] + VI3G1R2*PERF[I3,R2] ...) / (VI1G1R2 + VI2G1R2 + VI3G1R2 + ...)
# BADG2R2 => (VI1G2R2*PERF[I1,R2] + VI2G2R2*PERF[I2,R2] + VI3G2R2*PERF[I3,R2] ...) / (VI1G2R2 + VI2G2R2 + VI3G2R2 + ...)
# BADG3R2 => (VI1G3R2*PERF[I1,R2] + VI2G3R2*PERF[I2,R2] + VI3G3R2*PERF[I3,R2] ...) / (VI1G3R2 + VI2G3R2 + VI3G3R2 + ...)
# BADG4R2 => (VI1G4R2*PERF[I1,R2] + VI2G4R2*PERF[I2,R2] + VI3G4R2*PERF[I3,R2] ...) / (VI1G4R2 + VI2G4R2 + VI3G4R2 + ...)

# BADG1R3 => (VI1G1R3*PERF[I1,R3] + VI2G1R3*PERF[I2,R3] + VI3G1R3*PERF[I3,R3] ...) / (VI1G1R3 + VI2G1R3 + VI3G1R3 + ...)
# BADG2R3 => (VI1G2R3*PERF[I1,R3] + VI2G2R3*PERF[I2,R3] + VI3G2R3*PERF[I3,R3] ...) / (VI1G2R3 + VI2G2R3 + VI3G2R3 + ...)
# BADG3R3 => (VI1G3R3*PERF[I1,R3] + VI2G3R3*PERF[I2,R3] + VI3G3R3*PERF[I3,R3] ...) / (VI1G3R3 + VI2G3R3 + VI3G3R3 + ...)
# BADG4R3 => (VI1G4R3*PERF[I1,R3] + VI2G4R3*PERF[I2,R3] + VI3G4R3*PERF[I3,R3] ...) / (VI1G4R3 + VI2G4R3 + VI3G4R3 + ...)

# BADG1R1 = Z11 ==> CONSTRAINT ==> Z11 = SUM(assignment_vars[(1,1,idx)]*PERF[(1,idx)] for idx) / SUM(assignment_vars[(1,1,idx)])
#                                  SUM(assignment_vars[(1,1,idx)]) * Z11 = SUM(assignment_vars[(1,1,idx)]*PERF[(1,idx)] for idx)

#                            U11 = SUM(assignment_vars[(1,1,idx)]) * Z11 ==> U11 = SUM(assignment_vars[(1,1,idx)]*PERF[(1,idx)] for idx)
#                                                                            U11 >= 0*Z11
#                                                                            U11 <= SUM(PERF[(1,idx)] for idx)*Z11

# -----------------------------------------------------------------------------------------------------------------------

# MEDG1 => (BADG1R1 + BADG1R2 + BADG1R3) * INVERSO_TOTAL_REFS
# MEDG2 => (BADG2R1 + BADG2R2 + BADG2R3) * INVERSO_TOTAL_REFS
# MEDG3 => (BADG3R1 + BADG3R2 + BADG3R3) * INVERSO_TOTAL_REFS
# MEDG4 => (BADG4R1 + BADG4R2 + BADG4R3) * INVERSO_TOTAL_REFS

# MEDG1 = Y1 ==> CONSTRAINT ==> TOTAL_REFS * Y1 = SOMA(Z11,Z12,Z13) 
# -----------------------------------------------------------------------------------------------------------------------

# STDDEVG1 => SOMA([|BADG1R1-MEDG1|,|BADG1R2-MEDG1|,|BADG1R3-MEDG1|]) * INVERSO_TOTAL_REFS
# STDDEVG2 => SOMA([|BADG2R1-MEDG2|,|BADG2R2-MEDG2|,|BADG2R3-MEDG2|]) * INVERSO_TOTAL_REFS
# STDDEVG3 => SOMA([|BADG3R1-MEDG3|,|BADG3R2-MEDG3|,|BADG3R3-MEDG3|]) * INVERSO_TOTAL_REFS
# STDDEVG4 => SOMA([|BADG4R1-MEDG4|,|BADG4R2-MEDG4|,|BADG4R3-MEDG4|]) * INVERSO_TOTAL_REFS

# STDDEVG1 = X1 ==> CONSTRAINT ==> TOTAL_REFS * X1 = (|Z11-Y1| + |Z12-Y1| + |Z13-Y1|)
# |Z11-Y1| = W11 ==> CONSTRAINT ==> TOTAL_REFS * X1 = (W11 + W12 + W13) 
#                                    Z11-Y1 <= W11
#                                    Y1-Z11 <= W11
# -----------------------------------------------------------------------------------------------------------------------

# MIN(STDDEVG1+STDDEVG2+STDDEVG3+STDDEVG4)

# MIN(X1+X2+X3+X4)

In [45]:
def generate_best_gh_problem(df, ref_var, perf_var, n_groups):
    refs_list = sorted([sfr for sfr in df[ref_var].unique()])
    groups_list = sorted([grp for grp in np.arange(1,n_groups+1)])
    idx_dict = {ref:[i for i in range(1,len(df[df[ref_var]==ref]) + 1)] for ref in refs_list}

    ref_data_dict = {ref:df[df[safra]==ref][[perf,score]].sort_values(score).reset_index(drop=True) for ref in refs_list}
    ref_idx_perf_dict = {(ref,i+1):int(df_perf_score.loc[i][perf])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
    ref_idx_score_dict = {(ref,i+1):float(df_perf_score.loc[i][score])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
    
    n_idx_refs_dict = {ref:len(idx_dict[ref]) for ref in idx_dict}

    assignment_vars = {} # VIxGxRx
    for ref in refs_list:
        for grp in groups_list:            
            for idx in idx_dict[ref]:
                if idx < 10:
                    idx_str = f'0{idx}'
                else:
                    idx_str = f'{idx}'
                    
                assignment_vars[(ref,grp,idx)] = LpVariable(f'ref_{ref}_grp_{grp}_idx_{idx_str}', cat=LpBinary)

    
    qtd_maus_grp_vars = LpVariable.dicts("qtd_maus_ref_grp", (refs_list, groups_list), 0, None, LpInteger)
    
    bad_vars = LpVariable.dicts("bad_ref_grp", (refs_list, groups_list), 0, 1, LpContinuous)

    med_vars = LpVariable.dicts("media_grp", groups_list, 0, None, LpContinuous)

    desvio_absoluto_vars = LpVariable.dicts("desvio_absoluto_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)
    
    desvio_medio_vars = LpVariable.dicts("desvio_medio_grp", groups_list, 0, None, LpContinuous)

    

    
    prob = LpProblem("BestGH", LpMinimize)
    
    prob += (
        lpSum([desvio_medio_vars[grp] for grp in groups_list]),
        "SomadosDesviosMediosdasBadRatesDosGHsAoLongoDasRefs",
    )
    
    # RESTRIÇÕES DO DESVIO MÉDIO
    for grp in groups_list:
        prob += (
            len(refs_list)*desvio_medio_vars[grp] == lpSum([desvio_absoluto_vars[ref][grp] for ref in refs_list]),
            f"DefinicaoDesvioMedio_grp_{grp}",
        )

    # RESTRIÇÕES DO DESVIO ABSOLUTO
    for grp in groups_list:
        for ref in refs_list:
            prob += (
                bad_vars[ref][grp] - med_vars[grp] <= desvio_absoluto_vars[ref][grp],
                f"DefinicaoDesvioAbsoluto(1)_grp_{grp}_ref_{ref}"
            )

    for grp in groups_list:
        for ref in refs_list:
            prob += (
                med_vars[grp] - bad_vars[ref][grp] <= desvio_absoluto_vars[ref][grp],
                f"DefinicaoDesvioAbsoluto(2)_grp_{grp}_ref_{ref}"
            )


    
    # RESTRIÇÕES DA MÉDIA
    for grp in groups_list:
        prob += (
            len(refs_list)*med_vars[grp] == lpSum([bad_vars[ref][grp] for ref in refs_list]),
            f"DefinicaoMedia_grp_{grp}",
        )
        

    # RESTRIÇÕES DAS VARIÁVEIS DE ATRIBUIÇÃO
    for grp in groups_list:
        for ref in refs_list:
            prob += (
                qtd_maus_grp_vars[ref][grp]  == lpSum([assignment_vars[ref,grp,idx]*ref_idx_perf_dict[(ref,idx)] for idx in idx_dict[ref]]) ,
                f"DefinicaoVariaveisAtribuicao(1)_grp_{grp}_ref_{ref}",
            )

    for grp in groups_list:
        for ref in refs_list:
            prob += (
                qtd_maus_grp_vars[ref][grp] >= 0,
                f"DefinicaoVariaveisAtribuicao(2)_grp_{grp}_ref_{ref}",
            )

    for grp in groups_list:
        for ref in refs_list:
            prob += (
                qtd_maus_grp_vars[ref][grp] <= lpSum([ref_idx_perf_dict[(ref,idx)] for idx in idx_dict[ref]]) * bad_vars[ref][grp],
                f"DefinicaoVariaveisAtribuicao(3)_grp_{grp}_ref_{ref}",
            )   
    



    
    # RESTRIÇÃO ATRIBUIÇÃO A SOMENTE 1 GRUPO
    for ref in refs_list:
        for idx in idx_dict[ref]:
            prob += (
                lpSum([assignment_vars[ref,grp,idx] for grp in groups_list]) == 1,
                f"CadaPontoAtribuidoASomente1Grupo_ref_{ref}_idx_{idx}",
            )    

    # RESTRIÇÃO ATRIBUIÇÃO A PELO MENOS 1 GRUPO
    for ref in refs_list:
        for grp in groups_list:    
            prob += (
                lpSum([assignment_vars[ref,grp,idx] for idx in idx_dict[ref]]) >= 2,
                f"CadaGrupoPeloMenos1Membro_r{ref}_g{grp}_i{idx}",
            )

    # RESTRIÇÃO DE COMPLETUDE: PARA CADA REF, TODO E CADA UM DOS INDICES É ATRIBUÍDO A UM GRUPO            
    for ref in refs_list:    
        prob += (
            lpSum([assignment_vars[ref,grp,idx] for idx in idx_dict[ref] for grp in groups_list]) == len(idx_dict[ref]),
            f"TotalIndicesPorRef_r{ref}",
        )

    # RESTRIÇÃO DE ORDENAÇÃO DAS MÉDIAS: A MÉDIA DOS VALORES DE BAD RATE AO LONGO DAS REFS PARA O GRUPO 1 DEVE SER MENOR QUE A DO GRUPO 2. A DO GRUPO 2, MENOR QUE A DO GRUPO 3 E ETC...
    for grp in groups_list[:-1]:
        prob += (
            med_vars[grp] <= med_vars[grp+1],
            f"OrdemMediaGrupos_grp_{grp}"
        )

    # RESTRIÇÃO DE ORDENAÇÃO DAS BAD RATES: PARA UMA MESMA REF, O VALOR DE BAD RATE DO GRUPO 1 DEVE SER MENOR QUE O DO GRUPO 2. O DO GRUPO 2, MENOR QUE O DO GRUPO 3 E ETC...
    for ref in refs_list:
        for grp in groups_list[:-1]:    
            prob += (
                bad_vars[ref][grp] <= bad_vars[ref][grp+1],
                f"OrdemBadGrupos_grp_{grp}_ref_{ref}"
            )

    # RESTRIÇÃO DE NÃO CRUZAMENTO DAS BAD RATES: PARA UMA DETERMINADA REF, A BAD RATE DO GRUPO 1 DEVE SER SEMPRE MENOR QUE A BAD RATE DO GRUPO 2 PARA TODAS AS DEMAIS REFS. A DO GRUPO 2, MENOR QUE A DO GRUPO 3 E ETC...
    for ref in refs_list:
        #other_refs_list = list(set(refs_list.copy()) - set(ref))
        for grp in groups_list[:-1]:    
            for ref2 in refs_list:
                prob += (
                    bad_vars[ref][grp] <= bad_vars[ref2][grp+1],
                    f"NaoCruzamentoBadGrupos_g{grp}_r{ref}_r{ref2}"
                )

    
    # RESTRIÇÃO SCORES ORDENADOS: INDICE 1 POSSUI O MENOR VALOR DE SCORE E INDICE N POSSUI O MAIOR
    for ref in refs_list:
        for idx in idx_dict[ref][:-1]:            
            prob += (
                lpSum([assignment_vars[ref,grp,idx] for grp in groups_list]) * ref_idx_score_dict[(ref,idx)] <= lpSum([assignment_vars[ref,grp,idx+1] for grp in groups_list]) * ref_idx_score_dict[(ref,idx+1)],
                f"ScoresPermanecemOrdenados_ref_{ref}_idx_{idx}",
            )


    # # RESTRIÇÃO DE ORDEM DE PREENCHIMENTO DOS GRUPOS: EM CADA REF, OS MENORES INDICES PERTENCEM AO 1o GRUPO, OS SEGUINTES AO 2o, etc...
    # for ref in refs_list:
    #     for i, grp in enumerate(groups_list,start=1):
    #         for idx in idx_dict[ref]:
    #             prob += (
    #                 assignment_vars[ref,grp,idx] >= lpSum([assignment_vars[ref,grp,idx] for grp in groups_list]) - big_M*(1-assignment_vars[ref,grp,idx]),
    #                 f"PreencherGruposMenorProMaiorIndice_r{ref}_g{grp}_i{idx}",
    #             )

    # RESTRIÇÃO DE ORDEM DE PREENCHIMENTO DOS GRUPOS: EM CADA REF, OS MENORES INDICES PERTENCEM AO 1o GRUPO, OS SEGUINTES AO 2o, etc...
    for ref in refs_list:
        big_M = n_idx_refs_dict[ref]
        for grp in groups_list:
            for i, idx in enumerate(idx_dict[ref][:-1]):
                # print(f'idx = {idx}')
                # print(f'idx + 1 = {idx+1}')
                # print(f'idx maiores = {[idx2 for idx2 in idx_dict[ref][(i+1):]]}')
                prob += (
                    assignment_vars[ref,grp,idx] - 1 + big_M*(1 - assignment_vars[ref,grp,idx])  >= lpSum([assignment_vars[ref,grp,idx2] for idx2 in idx_dict[ref][(i+1):]]) - big_M*(assignment_vars[ref,grp,idx+1]) ,
                    f"PreencherGruposMenorProMaiorIndice(1)_r{ref}_g{grp}_i{idx}",
                )# VRGI1 + big_M*(1 - VRGI1)  >= (VRGI2+VRGI3+...+VRGIN) - big_M*(VRGI2)
                 # - Se VRGI1 == 0 e VRGI2 == 0   ==>   0 + big_M >=  (VRGI2+VRGI3+...+VRGIN) - 0
                 #                                ==>   big_M >=  (VRGI2+VRGI3+...+VRGIN)
                )# - Se VRGI1 == 0 e VRGI2 == 1   ==>   0 + big_M >=  (VRGI2+VRGI3+...+VRGIN) - 0
                 #                                ==>   big_M >=  (VRGI2+VRGI3+...+VRGIN)
    for ref in refs_list:
        big_M = n_idx_refs_dict[ref]
        for grp in groups_list:
            for i, idx in enumerate(idx_dict[ref][:-1]):
                prob += (
                    assignment_vars[ref,grp,idx+1] - 1 + big_M*(assignment_vars[ref,grp,idx])  >= lpSum([assignment_vars[ref,grp,idx2] for idx2 in idx_dict[ref][:(i+1)]]) - big_M*(1 - assignment_vars[ref,grp,idx+1]) ,
                    f"PreencherGruposMenorProMaiorIndice(2)_r{ref}_g{grp}_i{idx}",
                )

    
    # RESTRIÇÃO DE ORDEM DE PREENCHIMENTO DOS GRUPOS: EM CADA REF, O 1o GRUPO É PREENCHIDO PELOS MENORES INDICES, EM SEGUIDA O 2o GRUPO É PREENCHIDO COM OS INDICES SUBSEQUENTES, ETC...
    for ref in refs_list:
        for idx in idx_dict[ref]:
            for i, grp in enumerate(groups_list[:-1]):
                prob += (
                    assignment_vars[ref,grp,idx] - 1 + n_groups*(1 - assignment_vars[ref,grp,idx]) >= lpSum([assignment_vars[ref,grp2,idx] for grp2 in groups_list[(i+1):]]),
                    f"PreencherGruposMenorProMaiorGrupo(1)_r{ref}_g{grp}_i{idx}",
                )


    for ref in refs_list:
        for idx in idx_dict[ref]:
            for i, grp in enumerate(groups_list[:-1]):
                prob += (
                    assignment_vars[ref,grp+1,idx] - 1 + n_groups*(assignment_vars[ref,grp,idx]) >= lpSum([assignment_vars[ref,grp2,idx] for grp2 in groups_list[(i+1):]]),
                    f"PreencherGruposMenorProMaiorGrupo(2)_r{ref}_g{grp}_i{idx}",
                )

    
    

    for ref in refs_list:    
        assignment_vars[ref,1,1].setInitialValue(1)
        assignment_vars[ref,1,1].fixValue()

    
    return prob, assignment_vars



    # for ref in refs_list:
    #     for grp in groups_list:
    #         for idx in idx_dict[ref][:-1]:
    #             # RESTRIÇÃO PRRENCHER OS GRUPOS DO MENOR AO MAIOR SCORE
    #             prob += (
    #                 assignment_vars[ref,grp,idx] + big_M*(1 - assignment_vars[ref,grp,idx]) >= assignment_vars[ref,grp,str(int(idx)+1)],
    #                 f"PreencherGruposMenorProMaiorIndice_r{ref}_g{grp}_i{idx}",
    #             )    
    

    # # O GRUPO DO INDICE i É MENOR QUE O GRUPO DO ÍNDICE i+1 PARA TODO ÍNDICE PARA UMA MESMA REF
    # for ref in refs_list:
    #     for grp in groups_list:
    #         for idx in idx_dict[ref][:-1]:
    #             # RESTRIÇÃO PRRENCHER OS GRUPOS DO MENOR AO MAIOR SCORE
    #             prob += (
    #                 int(grp)*assignment_vars[ref,grp,idx] <= (int(grp))*assignment_vars[ref,grp,str(int(idx)+1)],
    #                 f"PreencherGruposMenorProMaiorIndice_r{ref}_g{grp}_i{idx}",
    #             )
    
    # for ref in refs_list:
    #     for idx in idx_dict[ref]:
    #         for grp in groups_list[:-1]:
    #             prob += (
    #                 assignment_vars[ref,grp,idx] + big_M*(1 - assignment_vars[ref,grp,idx]) >= lpSum([assignment_vars[ref,str(grp),idx] for grp in groups_list]),
    #                 f"PreencherGruposMenorProMaiorGrupo_r{ref}_g{grp}_i{idx}",
    #             )

    # for ref in refs_list:
    #     for idx in idx_dict[ref]:
    #         for grp in groups_list[:-1]:
    #             prob += (
    #                 idx*assignment_vars[ref,grp,idx] + big_M*(1 - assignment_vars[ref,grp,idx]) <= assignment_vars[ref,str(int(grp)+1),idx],
    #                 f"PreencherGruposMenorProMaiorGrupo_r{ref}_g{grp}_i{idx}",
    #             )

    # # TODO INDICE DO G1 É MENOR QUE TODO INDICE DO G2 PARA UMA MESMA REF 
    # for ref in refs_list:
    #     for idx in idx_dict[ref]:
    #         for grp in groups_list[:-1]:
    #             prob += (
    #                 int(idx)*assignment_vars[ref,grp,idx] <= int(idx)*assignment_vars[ref,str(int(grp)+1),idx],
    #                 f"PreencherGruposMenorProMaiorGrupo_r{ref}_g{grp}_i{idx}",
    #             )
    
    # for ref in refs_list:
    #     for idx in idx_dict[ref][:-1]:            
    #         # RESTRIÇÃO ÍNDICES ORDENADOS
    #         prob += (
    #             lpSum([assignment_vars[ref,grp,idx] for grp in groups_list]) <= lpSum([assignment_vars[ref,grp,str(int(idx)+1)] for grp in groups_list]) * ref_idx_score_dict[(ref,str(int(idx)+1))],
    #             f"ScoresPermanecemOrdenados_ref_{ref}_idx_{idx}",
    #         )

    # # O GRUPO DO INDICE i É MENOR QUE O GRUPO DO ÍNDICE i+1 PARA TODO ÍNDICE PARA UMA MESMA REF
    # for ref in refs_list:
    #     for grp in groups_list:
    #         for i, idx in enumerate(idx_dict[ref][:-1]):
    #             # RESTRIÇÃO PRRENCHER OS GRUPOS DO MENOR AO MAIOR SCORE
    #             prob += (
    #                 assignment_vars[ref,grp,idx] + len(idx_dict[ref])*(1 - assignment_vars[ref,grp,idx]) >= lpSum([assignment_vars[ref,grp,idx] for idx in idx_dict[ref][(i+1):]]),
    #                 f"PreencherGruposMenorProMaiorIndice_r{ref}_g{grp}_i{idx}",
    #             )

    # for ref in refs_list:
    #     for idx in idx_dict[ref][:-1]:
    #         for i, grp in enumerate(groups_list[:-1]):
    #             prob += (
    #                 lpSum([assignment_vars[ref,grp,idx] for grp in groups_list[i:]]) >= lpSum([assignment_vars[ref,grp,idx] for grp in groups_list[i+1:]]),
    #                 f"PreencherGruposMenorProMaiorGrupo_r{ref}_g{grp}_i{idx}",
    #             )



    
    # for ref in refs_list:
    #     for grp in groups_list:
    #         for i, idx in enumerate(idx_dict[ref][:-1]):
    #             # RESTRIÇÃO PRRENCHER OS GRUPOS DO MENOR AO MAIOR SCORE
    #             prob += (
    #                 assignment_vars[ref,grp,idx] + 100*(1-assignment_vars[ref,grp,idx]) >= assignment_vars[ref,grp,idx+1],
    #                 f"PreencherGruposMenorProMaiorIndice_r{ref}_g{grp}_i{idx}",
    #             )

In [54]:
#     # VIXG1RX + VIXG2RX + VIXG3RX + VIXG4RX = 1 PARA TODO I, PARA TODO R
#     # (VI1G1RX + VI1G2RX + VI1G3RX + VI1G4RX)*SCORE[Ix,RX] < (VIx+1G1RX + VIx+1G2RX + VI2G3RX + VI2G4RX)*SCORE[Ix+1,RX] PARA TODO I, PARA TODO R
#     # VI2GxRx < 1 - VI1GxRx PARA TODO G PARA TODO R
#     # VIxG2Rx < 1 - VIxG1Rx PARA TODO I PARA TODO R


# if score1 < score2 then grp1 < grp2

#                  Na transicao de grupo == 1 senão 0
# IDX TROCA GRUPO = (VIyG1Rx-VIy+1G1Rx)*(Iy) todos indicadores abaixo do indice y são maiores que todos acima
# (I2)*VI2G1Rx >= (I3)*VI3G2Rx

# o indice dos V do grupo 1 são menores que o indice dos V do grupo 2

In [55]:
# y1 é IDX TROCA GRUPO1 = (VIyG1R-VIy+1G1R)*(Iy)
# y2 é IDX TROCA GRUPO2 = (VIyG2R-VIy+1G2R)*(Iy)
# y3 é IDX TROCA GRUPO3 = (VIyG3R-VIy+1G3R)*(Iy)
# y4 é IDX TROCA GRUPO4 = (VIyG4R-VIy+1G4R)*(Iy)

# se I < y1 ==> VIyG1R > VIyG2R
#               VIyG1R > VIyG3R
#               VIyG1R > VIyG4R
# se y1 < I < y2 ==> VIyG2R > VIyG3R
#                    VIyG2R > VIyG4R
# se y2 < I < y3 ==> VIyG3R > VIyG4R


# for i in idx_list:

# soma(VIxG1R + VIxG2R + VIxG3R + VIxG4R) - 1  <= 1*(VIxG1R)

# soma(VIxG2R + VIxG3R + VIxG4R) <= (VIxG1R)

In [56]:
# X = desvio médio
# W = desvio absoluto
# Y = media
# Z = bad
# V = vars atribuicao
# U = numerador bad

In [46]:
safra = 'SAFRA'
perf = 'PERF'
score = 'SCORE'
gh = 'GH'
index = 'INDEX'
n_groups = 4

In [47]:
prob, assignment_vars = generate_best_gh_problem(df, safra, perf, n_groups)

# refs_list = sorted([int(sfr) for sfr in df[safra].unique()])
# groups_list = sorted([int(grp) for grp in np.arange(1,n_groups+1)])
# idx_dict = {ref:[int(i) for i in range(0,len(df[df[safra]==int(ref)]))] for ref in refs_list}

refs_list = sorted([int(sfr) for sfr in df[safra].unique()])
groups_list = sorted([int(grp) for grp in np.arange(1,n_groups+1)])
idx_dict = {ref:[i for i in range(1,len(df[df[safra]==ref]) + 1)] for ref in refs_list}

ref_data_dict = {ref:df[df[safra]==ref][[perf,score]].sort_values(score).reset_index(drop=True) for ref in refs_list}
ref_idx_perf_dict = {(ref,i+1):int(df_perf_score.loc[i][perf])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
ref_idx_score_dict = {(ref,i+1):float(df_perf_score.loc[i][score])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}

# ref_data_dict = {ref:df[df[safra]==int(ref)][[perf,score]].sort_values(score).reset_index(drop=True) for ref in refs_list}
# ref_idx_perf_dict = {(ref,int(i)):int(df_perf_score.loc[i][perf])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
# ref_idx_score_dict = {(ref,int(i)):float(df_perf_score.loc[i][score])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
#ref_idx_perf_dict
#desvio_absoluto_vars = LpVariable.dicts("desvio_absoluto_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)

In [48]:
prob

BestGH:
MINIMIZE
1*desvio_medio_grp_1 + 1*desvio_medio_grp_2 + 1*desvio_medio_grp_3 + 1*desvio_medio_grp_4 + 0
SUBJECT TO
DefinicaoDesvioMedio_grp_1: - desvio_absoluto_ref_grp_2306_1
 - desvio_absoluto_ref_grp_2307_1 - desvio_absoluto_ref_grp_2308_1
 + 3 desvio_medio_grp_1 = 0

DefinicaoDesvioMedio_grp_2: - desvio_absoluto_ref_grp_2306_2
 - desvio_absoluto_ref_grp_2307_2 - desvio_absoluto_ref_grp_2308_2
 + 3 desvio_medio_grp_2 = 0

DefinicaoDesvioMedio_grp_3: - desvio_absoluto_ref_grp_2306_3
 - desvio_absoluto_ref_grp_2307_3 - desvio_absoluto_ref_grp_2308_3
 + 3 desvio_medio_grp_3 = 0

DefinicaoDesvioMedio_grp_4: - desvio_absoluto_ref_grp_2306_4
 - desvio_absoluto_ref_grp_2307_4 - desvio_absoluto_ref_grp_2308_4
 + 3 desvio_medio_grp_4 = 0

DefinicaoDesvioAbsoluto(1)_grp_1_ref_2306: bad_ref_grp_2306_1
 - desvio_absoluto_ref_grp_2306_1 - media_grp_1 <= 0

DefinicaoDesvioAbsoluto(1)_grp_1_ref_2307: bad_ref_grp_2307_1
 - desvio_absoluto_ref_grp_2307_1 - media_grp_1 <= 0

DefinicaoDesvioAbs

In [49]:
prob.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/thommic/Python/venvs/best_gh/lib64/python3.12/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/97c5807081314fd2a7ed6590f588f786-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/97c5807081314fd2a7ed6590f588f786-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1988 COLUMNS
At line 26743 RHS
At line 28727 BOUNDS
At line 29224 ENDATA
Problem MODEL has 1983 rows, 516 columns and 23428 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.01 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.02   (Wallclock seconds):       0.02



-1

In [36]:
for var in prob.variables():
    print(f'Var: {var} | Value: {var.value()}')

Var: bad_ref_grp_2306_1 | Value: 0.5
Var: bad_ref_grp_2306_2 | Value: 0.5
Var: bad_ref_grp_2306_3 | Value: 0.5
Var: bad_ref_grp_2306_4 | Value: 0.5
Var: bad_ref_grp_2307_1 | Value: 0.5
Var: bad_ref_grp_2307_2 | Value: 0.5
Var: bad_ref_grp_2307_3 | Value: 0.5
Var: bad_ref_grp_2307_4 | Value: 0.5
Var: bad_ref_grp_2308_1 | Value: 0.5
Var: bad_ref_grp_2308_2 | Value: 0.5
Var: bad_ref_grp_2308_3 | Value: 0.5
Var: bad_ref_grp_2308_4 | Value: 0.5
Var: desvio_absoluto_ref_grp_2306_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_3 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_4 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_3 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_4 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_3 | Value: 0.0
Var: desvio_absol

In [37]:
for const in prob.constraints:
    print(f'Constraint: {const} | Value: {prob.constraints[const].value()}')

Constraint: DefinicaoDesvioMedio_grp_1 | Value: 0.0
Constraint: DefinicaoDesvioMedio_grp_2 | Value: 0.0
Constraint: DefinicaoDesvioMedio_grp_3 | Value: 0.0
Constraint: DefinicaoDesvioMedio_grp_4 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_1_ref_2306 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_1_ref_2307 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_1_ref_2308 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_2_ref_2306 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_2_ref_2307 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_2_ref_2308 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_3_ref_2306 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_3_ref_2307 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_3_ref_2308 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_4_ref_2306 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_4_ref_2307 | Value: 0.0
Constraint: DefinicaoDesvioAbsoluto(1)_grp_4_ref_2308 |

In [38]:
prob.constraints[list(prob.constraints.keys())[0]].value()

0.0

In [39]:
solution_dict = {}

for ref in refs_list:
    solution_dict[int(ref)] = {}
    
    for grp in groups_list:
        solution_dict[int(ref)][int(grp)] = []
        
        for idx in idx_dict[ref]:
            if assignment_vars[int(ref),int(grp),int(idx)].value() == 1:
                solution_dict[int(ref)][int(grp)].append(idx)

In [40]:
solution_dict

{2306: {1: [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34],
  2: [35, 36],
  3: [39, 40],
  4: [37, 38]},
 2307: {1: [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34],
  2: [39, 40],
  3: [35, 36],
  4: [37, 38]},
 2308: {1: [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27],
  2: [28, 29],
  3: [30, 31, 32, 33, 34, 35, 36],
  4: [37, 38]}}

In [41]:
grp_ref_bad_rate = {}
grp_bad_mean = {}
grp_std_dev = {}

for ref in refs_list:
    grp_ref_bad_rate[ref] = {}
    for grp in groups_list:
        grp_ref_bad_rate[int(ref)][int(grp)] = []
        for idx in solution_dict[int(ref)][int(grp)]:
            grp_ref_bad_rate[int(ref)][int(grp)].append(ref_idx_perf_dict[ref,idx])
        grp_ref_bad_rate[int(ref)][int(grp)] = 0 if len(grp_ref_bad_rate[int(ref)][int(grp)]) == 0 else float(np.sum(grp_ref_bad_rate[int(ref)][int(grp)])/len(grp_ref_bad_rate[int(ref)][int(grp)]))

for grp in groups_list:
    grp_bad_mean[int(grp)] = []
    for ref in refs_list:
        grp_bad_mean[int(grp)].append(grp_ref_bad_rate[int(ref)][int(grp)])
    grp_bad_mean[int(grp)] = float(np.sum(grp_bad_mean[int(grp)])/len(grp_bad_mean[int(grp)]))


for grp in groups_list:
    grp_std_dev[int(grp)] = []
    for ref in refs_list:
        grp_std_dev[int(grp)].append(grp_ref_bad_rate[int(ref)][int(grp)])
    grp_std_dev[int(grp)] = float(np.std(grp_std_dev[int(grp)]))

# for grp in groups_list:
#     grp_std_dev[grp] = []
#     for ref in refs_list:
#         grp_std_dev[grp].append(grp_bad_mean[grp])
#     grp_std_dev[grp] = np.std(grp_std_dev[grp])

# STDDEVG1 => SOMA([|BADG1R1-MEDG1|,|BADG1R2-MEDG1|,|BADG1R3-MEDG1|]) * INVERSO_TOTAL_REFS
# ref_idx_perf_dict = {(ref,str(i)):int(df_perf_score.loc[i][perf])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
# ref_idx_score_dict = {(ref,str(i)):float(df_perf_score.loc[i][score])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}

In [ ]:
grp_ref_bad_rate

In [43]:
grp_bad_mean

{1: 0.08605664488017428, 2: 0.5, 3: 0.5476190476190476, 4: 0.5}

In [44]:
grp_std_dev

{1: 0.021401833011316446,
 2: 0.408248290463863,
 3: 0.35154340619603336,
 4: 0.408248290463863}

In [139]:
refs_list = sorted([str(sfr) for sfr in df[safra].unique()])
groups_list = sorted([str(grp) for grp in np.arange(1,n_groups+1)])
idx_dict = {ref:[str(i) for i in range(0,len(df[df[safra]==int(ref)]))] for ref in refs_list}

ref_data_dict = {ref:df[df[safra]==int(ref)][[perf,score]].sort_values(score).reset_index(drop=True) for ref in refs_list}
ref_idx_perf_dict = {(ref,str(i)):int(df_perf_score.loc[i][perf])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}
ref_idx_score_dict = {(ref,str(i)):float(df_perf_score.loc[i][score])  for ref,df_perf_score in ref_data_dict.items() for i in df_perf_score.index}

#inverse_total_grps = 1/n_groups
n_idx_refs_dict = {ref:len(idx_dict[ref]) for ref in idx_dict}


assignment_vars = {} # VIxGxRx
#order_indicator_vars = {} # T
for ref in refs_list:
    for grp in groups_list:
        #zip_idx = [(i,x) for i,x in zip(idx_dict[ref][:-1],idx_dict[ref][1:])]
        
        for idx in idx_dict[ref]:
            assignment_vars[(str(ref),str(grp),str(idx))] = LpVariable(f'ref_{ref}_grp_{grp}_idx_{idx}', cat=LpBinary)

        #for idx1,idx2 in zip_idx:
        #    order_indicator_vars[(ref,grp,idx1,idx2)] = LpVariable(f'ref_{ref}_grp_{grp}_T_{idx1}_{idx2}', cat=LpBinary)

qtd_maus_grp_vars = LpVariable.dicts("qtd_maus_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)

bad_vars = LpVariable.dicts("bad_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)

med_vars = LpVariable.dicts("media_grp", groups_list, 0, None, LpContinuous)

desvio_absoluto_vars = LpVariable.dicts("desvio_absoluto_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)

desvio_medio_vars = LpVariable.dicts("desvio_medio_grp", groups_list, 0, None, LpContinuous)

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from pulp import *

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

In [10]:
ref = [2306]*80 + [2307]*80 + [2308]*76
perf = [0]*100 + [1]*4 + [0]*20 + [1]*4 + [0]*20 + [1]*4 + [0]*20 + [1]*4 + [0]*20 + [1]*4 + [0]*20 + [1]*16
score = [float(scr) for scr in np.linspace(0,1,236)]

In [11]:
len(ref), len(perf), len(score)

(236, 236, 236)

In [12]:
np.random.seed(42)
np.random.shuffle(ref)
ref[:10]

[2306, 2308, 2308, 2306, 2307, 2307, 2308, 2307, 2307, 2306]

In [13]:
df = pd.DataFrame({'SAFRA': ref, 'PERF': perf, 'SCORE': score})

In [14]:
df.SAFRA.value_counts()

SAFRA
2306    80
2307    80
2308    76
Name: count, dtype: int64

In [15]:
df.groupby('SAFRA').SCORE.mean()

SAFRA
2306    0.496649
2307    0.486968
2308    0.517245
Name: SCORE, dtype: float64

In [16]:
df.groupby('SAFRA').PERF.mean()

SAFRA
2306    0.112500
2307    0.125000
2308    0.223684
Name: PERF, dtype: float64

In [17]:
df.head()

,SAFRA,PERF,SCORE
0,2306,0,0.000000
1,2308,0,0.004255
2,2308,0,0.008511
3,2306,0,0.012766
4,2307,0,0.017021


In [19]:
safra = 'SAFRA'
perf = 'PERF'
score = 'SCORE'
gh = 'GH'
index = 'INDEX'
n_groups = 4

In [20]:
def generate_best_gh_problem(df, ref_var, perf_var, score_var, n_groups, pct_per_grp):
    refs_list = sorted([sfr for sfr in df[ref_var].unique()])
    groups_list = sorted([grp for grp in np.arange(1,n_groups+1)])
    idx_list = [i+1 for i in range(0,len(df[score]))]

    # n_maus_grp = ?
    # n_maus ao todo -> tenho
    
    n_maus_refs = {ref:int(df.groupby(ref_var)[perf_var].sum().loc[ref]) for ref in refs_list}
    n_idx_refs_dict = {ref:int(df[ref_var].value_counts().loc[ref]) for ref in refs_list}
    indicators_refs = {ref:np.where(df[ref_var] == ref, 1, 0) for ref in refs_list} 
    ref_idx_indicator_dict = {(ref,i+1):int(ind) for ref in refs_list for i, ind in enumerate(indicators_refs[ref])}     
    idx_perf_dict = {i+1:int(df[perf_var].loc[i]) for i in df.index}
    idx_score_dict = {i+1:float(df[score_var].loc[i]) for i in df.index}
    
    desvio_medio_vars = LpVariable.dicts("desvio_medio_grp", groups_list, 0, None, LpContinuous)
    desvio_absoluto_vars = LpVariable.dicts("desvio_absoluto_ref_grp", (refs_list, groups_list), 0, None, LpContinuous)
    med_vars = LpVariable.dicts("media_grp", groups_list, 0, 1, LpContinuous)
    qtd_maus_grp_vars = LpVariable.dicts("qtd_maus_ref_grp", (refs_list, groups_list), 0, len(idx_list), LpInteger)
    bad_vars = LpVariable.dicts("bad_ref_grp", (refs_list, groups_list), 0, 1, LpContinuous)
    
    assignment_vars = {} # VIxGx
    for grp in groups_list:            
        for idx in idx_list:
            if idx < 10:
                idx_str = f'0{idx}'
            else:
                idx_str = f'{idx}'
                
            assignment_vars[(grp,idx)] = LpVariable(f'g{grp}_i{idx_str}', cat=LpBinary)
    
    
    idx_cut_point_grp = {0:LpVariable("idx_cut_point_grp_0",0,len(idx_list),LpInteger)} | \
                         LpVariable.dicts("idx_cut_point_grp", groups_list, 0, len(idx_list), LpInteger)

    prob = LpProblem("BestGH", LpMinimize)
    
    prob += (
        lpSum([desvio_medio_vars[grp] for grp in groups_list]),
        "SomadosDesviosMediosdasBadRatesDosGHsAoLongoDasRefs",
    )



    
    
    # RESTRIÇÕES DO DESVIO MÉDIO
    for grp in groups_list:
        prob += (
            len(refs_list)*desvio_medio_vars[grp] == lpSum([desvio_absoluto_vars[ref][grp] for ref in refs_list]),
            f"DefinicaoDesvioMedio_g{grp}",
        )

    # RESTRIÇÕES DO DESVIO ABSOLUTO
    for grp in groups_list:
        for ref in refs_list:
            prob += (
                bad_vars[ref][grp] - med_vars[grp] <= desvio_absoluto_vars[ref][grp],
                f"DefinicaoDesvioAbsoluto(1)_g{grp}_r{ref}"
         )

    for grp in groups_list:
        for ref in refs_list:
            prob += (
                med_vars[grp] - bad_vars[ref][grp] <= desvio_absoluto_vars[ref][grp],
                f"DefinicaoDesvioAbsoluto(2)_g{grp}_r{ref}"
            )

    
    # RESTRIÇÕES DA MÉDIA
    for grp in groups_list:
        prob += (
            len(refs_list)*med_vars[grp] == lpSum([bad_vars[ref][grp] for ref in refs_list]),
            f"DefinicaoMedia_g{grp}",
        )
        

    # RESTRIÇÕES DAS VARIÁVEIS DE ATRIBUIÇÃO
    for grp in groups_list:
        for ref in refs_list:
            prob += (
                qtd_maus_grp_vars[ref][grp]  == lpSum([assignment_vars[grp,idx]*ref_idx_indicator_dict[(ref,idx)]\
                                                       *idx_perf_dict[idx] for idx in idx_list]) ,
                f"DefinicaoVariaveisAtribuicao(1)_g{grp}_r{ref}",
            )

    for grp in groups_list:
        for ref in refs_list:
            prob += (
                qtd_maus_grp_vars[ref][grp] <= lpSum([idx_perf_dict[idx]*ref_idx_indicator_dict[(ref,idx)] for idx in idx_list]) \
                                                        * bad_vars[ref][grp],
                f"DefinicaoVariaveisAtribuicao(2)_g{grp}_r{ref}",
            )

    
    # RESTRIÇÕES DOS CUT POINTS
    idx_cut_point_grp[0].setInitialValue(0)
    idx_cut_point_grp[0].fixValue()
    idx_cut_point_grp[n_groups].setInitialValue(len(idx_list))
    idx_cut_point_grp[n_groups].fixValue()
    for grp in groups_list:
        prob += (
            idx_cut_point_grp[grp]  >= lpSum([(assignment_vars[grp,idx] - assignment_vars[grp,idx+1])*idx for idx in idx_list[:-1]])\
                                        - idx_cut_point_grp[grp-1],
                f"DefinicaoCutPoints_g{grp}",
            )


    

    ###### AQUI COMEÇAM AS RESTRIÇÕES PROPRIAMENTE DITAS DO PROBLEMA
    
    # RESTRIÇÃO ATRIBUIÇÃO A SOMENTE 1 GRUPO
    for idx in idx_list:
        prob += (
            lpSum([assignment_vars[grp,idx] for grp in groups_list]) == 1,
            f"CadaPontoAtribuidoASomente1Grupo_idx_{idx}",
        )

    # # RESTRIÇÃO PELO MENOS 1 MEMBRO POR GRUPO EM CADA REF
    # for ref in refs_list:
    #     for grp in groups_list:    
    #         prob += (
    #             lpSum([assignment_vars[grp,idx]*ref_idx_indicator_dict[(ref,idx)] for idx in idx_list]) >= 1,
    #             f"CadaGrupoPeloMenos1Membro_r{ref}_g{grp}",
    #         )

    # RESTRIÇÃO PELO MENOS pct_per_grp% MEMBROS POR GRUPO
    for grp in groups_list:    
        prob += (
            lpSum([assignment_vars[grp,idx] for idx in idx_list]) >= pct_per_grp*len(idx_list),
            f"CadaGrupoPeloMenos%Membros_g{grp}",
        )

    # RESTRIÇÃO DE ORDENAÇÃO DAS MÉDIAS: A MÉDIA DOS VALORES DE BAD RATE AO LONGO DAS REFS PARA O GRUPO 1 DEVE SER MENOR 
    #   QUE A DO GRUPO 2. A DO GRUPO 2, MENOR QUE A DO GRUPO 3 E ETC...
    for grp in groups_list[:-1]:
        prob += (
            med_vars[grp] <= med_vars[grp+1],
            f"OrdemMediaGrupos_grp_{grp}"
        )


    # RESTRIÇÃO DE NÃO CRUZAMENTO DAS BAD RATES: PARA UMA DETERMINADA REF, A BAD RATE DO GRUPO 1 DEVE SER SEMPRE 
    #   MENOR QUE A BAD RATE DO GRUPO 2 PARA TODAS AS DEMAIS REFS. A DO GRUPO 2, MENOR QUE A DO GRUPO 3 E ETC...
    for ref in refs_list:
        for grp in groups_list[:-1]:    
            for ref2 in refs_list:
                prob += (
                    bad_vars[ref][grp] <= bad_vars[ref2][grp+1],
                    f"NaoCruzamentoBadGrupos_g{grp}_r{ref}_r{ref2}"
                )

   
    # RESTRIÇÃO DE COMPLETUDE: PARA CADA REF, TODO E CADA UM DOS INDICES É ATRIBUÍDO A UM GRUPO            
    for ref in refs_list:    
        prob += (
            lpSum([assignment_vars[grp,idx]*ref_idx_indicator_dict[(ref,idx)] for idx in idx_list for grp in groups_list])\
                == n_idx_refs_dict[ref],
            f"TotalIndicesPorRef_r{ref}",
        )

    
    # RESTRIÇÃO SCORES ORDENADOS: INDICE 1 POSSUI O MENOR VALOR DE SCORE E INDICE N POSSUI O MAIOR
    for idx in idx_list[:-1]:            
        prob += (
            lpSum([assignment_vars[grp,idx] for grp in groups_list]) * idx_score_dict[idx] <= \
            lpSum([assignment_vars[grp,idx+1] for grp in groups_list]) * idx_score_dict[idx+1],
            f"ScoresPermanecemOrdenados_i{idx}_i{idx+1}",
        )

    # RESTRIÇÃO PREENCHIMENTO ORDENADO DOS GRUPOS: OS MENORES INDICES PERTENCEM AO 1o GRUPO, OS SEGUINTES AO 2o, etc...
    for grp in groups_list:
        #for idx in idx_list:
        prob += (
            idx_cut_point_grp[grp-1] + lpSum([assignment_vars[grp,idx] for idx in idx_list]) <= idx_cut_point_grp[grp],
                f"PreencherGruposEntreOsCutPointsEncontrados_g{grp}_i{idx}",
            )


    # # RESTRIÇÃO DIFERENÇA ENTRE GRUPOS DE ÍNDICES CONSECUTIVOS É NO MÁXIMO 1
    # for ref in refs_list:
    #     idx_by_ref_list = [k[1] for k,v in ref_idx_indicator_dict.items() if k[0] == ref and v == 1]
    #     for i_idx,idx in enumerate(idx_by_ref_list[:-1]):
    #         prob += (
    #             # A DIFERENÇA ENTRE O GRUPO DO INDICE I+1 E O GRUPO DO INDICE I VALE NO MÁXIMO 1 (QUE EQUIVALE A DIFERENÇA DOS PRÓPRIOS INDICES)
    #             lpSum([assignment_vars[grp,idx_by_ref_list[i_idx+1]]*grp for grp in groups_list]) - 
    #             lpSum([assignment_vars[grp,idx]*grp for grp in groups_list]) <= 1,
    #                 f"ItensConsecutivosEmGruposConsecutivos_i{idx}_i{idx_by_ref_list[i_idx+1]}",
    #             )

    # RESTRIÇÃO DIFERENÇA ENTRE GRUPOS DE ÍNDICES CONSECUTIVOS É NO MÁXIMO 1
    for idx in idx_list[:-1]:
        prob += (
            # A DIFERENÇA ENTRE O GRUPO DO INDICE I+1 E O GRUPO DO INDICE I VALE NO MÁXIMO 1 (QUE EQUIVALE A DIFERENÇA DOS PRÓPRIOS INDICES)
            lpSum([assignment_vars[grp,idx+1]*grp for grp in groups_list]) - 
            lpSum([assignment_vars[grp,idx]*grp for grp in groups_list]) <= 1,
                f"ItensConsecutivosEmGruposConsecutivosMax_i{idx}_i{idx+1}"            
            )

    for idx in idx_list[:-1]:
        prob += (
            # A DIFERENÇA ENTRE O GRUPO DO INDICE I+1 E O GRUPO DO INDICE I VALE NO MÁXIMO 1 (QUE EQUIVALE A DIFERENÇA DOS PRÓPRIOS INDICES)
            lpSum([assignment_vars[grp,idx+1]*grp for grp in groups_list]) - 
            lpSum([assignment_vars[grp,idx]*grp for grp in groups_list]) >= 0,
                f"ItensConsecutivosEmGruposConsecutivosMin_i{idx}_i{idx+1}",
            )

    
    prim_idx_ref_dict = {ref:int(df[df[ref_var]==ref].index[0])+1 for ref in refs_list}
    ultm_idx_ref_dict = {ref:int(df[df[ref_var]==ref].index[-1])+1 for ref in refs_list}

    first_grp = 1
    for ref, idx in prim_idx_ref_dict.items():
        assignment_vars[first_grp,idx].setInitialValue(1)
        assignment_vars[first_grp,idx].fixValue()
    
    last_grp = groups_list[-1]
    for ref, idx in ultm_idx_ref_dict.items():
        assignment_vars[last_grp,idx].setInitialValue(1)
        assignment_vars[last_grp,idx].fixValue()
        
    
            
    # for ref in refs_list:    
    #     assignment_vars[ref,1,1].setInitialValue(1)
    #     assignment_vars[ref,1,1].fixValue()


    # for grp in groups_list:
    #     for ref in refs_list:
    #         prob += (
    #             qtd_maus_grp_vars[ref][grp] >= 0,
    #             f"DefinicaoVariaveisAtribuicao(2)_g{grp}_r{ref}",
    #         )


    
    # # RESTRIÇÃO DE ORDEM DE PREENCHIMENTO DOS GRUPOS: OS MENORES INDICES PERTENCEM AO 1o GRUPO, OS SEGUINTES AO 2o, etc...    
    # for grp in groups_list:
    #     for idx in idx_list[:-1]:
    #         big_M = len(idx_list) - idx
    #         prob += (
    #             big_M*(1 - assignment_vars[grp,idx]) >= lpSum([assignment_vars[grp,idx2] for idx2 in idx_list[idx:]]) 
    #             - big_M*(assignment_vars[grp,idx+1]) ,
    #             f"PreencherGruposMenorProMaiorIndice(1)_g{grp}_i{idx}")
    #         )# Quando IND[grp,idx] = 1 e IND[grp,idx+1] = 0, SUM(IND[grp,i]) = 0 para todo i > idx 
    #          #                                      big_M*(1 - VRGIn)  >= (VRGIn+1+VRGIn+2+...+VRGIN) - big_M*(VRGIn+1)
    #          # - Se VRGIn == 0 e VRGIn+1 == 0   ==>   big_M*1 >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M*0
    #          #                                  ==>     big_M >=  (VRGIn+1+VRGIn+2+...+VRGIN)

    #          # - Se VRGIn == 0 e VRGIn+1 == 1   ==>   big_M*1 >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M*1
    #          #                                  ==>     big_M >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M

    #          # - Se VRGIn == 1 e VRGIn+1 == 1   ==>   big_M*0 >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M*1
    #          #                                  ==>         0 >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M

    #          # - Se VRGIn == 1 e VRGIn+1 == 0   ==>   big_M*0 >=  (VRGIn+1+VRGIn+2+...+VRGIN) - big_M*0
    #          #                                  ==>         0 >=  (VRGIn+1+VRGIn+2+...+VRGIN)


    # for grp in groups_list:
    #     for idx in idx_list[:-1]:
    #         big_M = len(idx_list) - idx
    #         prob += (
    #             big_M*(assignment_vars[grp,idx])  >= lpSum([assignment_vars[grp,idx2] for idx2 in idx_list[:idx]]) - big_M*(1 - assignment_vars[grp,idx+1]) ,
    #             f"PreencherGruposMenorProMaiorIndice(2)_g{grp}_i{idx}")
    #         )# Quando IND[grp,idx] = 0 e IND[grp,idx+1] = 1, SUM(IND[grp,i]) = 0 para todo i < idx
    #          #                                    big_M*(VRGIn)  >= (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M*(1 - VRGIn+1)
    #          # - Se VRGI1 == 0 e VRGI2 == 0   ==>   big_M*0 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M*1
    #          #                                ==>         0 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M

    #          # - Se VRGI1 == 0 e VRGI2 == 1   ==>   big_M*0 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M*0
    #          #                                ==>         0 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1)

    #          # - Se VRGI1 == 1 e VRGI2 == 1   ==>   big_M*1 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M*0
    #          #                                ==>     big_M >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1)

    #          # - Se VRGI1 == 1 e VRGI2 == 0   ==>   big_M*1 >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M*1
    #          #                                ==>     big_M >=  (VRGIn+VRGIn-1+VRGIn-2+...+VRGI1) - big_M


    
    # RESTRIÇÃO DE ORDEM DE PREENCHIMENTO DOS GRUPOS: EM CADA REF, O 1o GRUPO É PREENCHIDO PELOS MENORES INDICES, EM SEGUIDA O 2o GRUPO É PREENCHIDO COM OS INDICES SUBSEQUENTES, ETC...
    # for idx in idx_list:
    #     for grp in groups_list[:-1]:   
    #         big_M = n_groups - grp
    #         prob += (
    #             big_M*(1 - assignment_vars[grp,idx])  >= lpSum([assignment_vars[grp2,idx] for grp2 in groups_list[grp:]]) - big_M*(1 - assignment_vars[grp+1,idx]) ,
    #             f"PreencherGruposMenorProMaiorGrupo(1)_g{grp}_i{idx}"
    #        )# Quando IND[grp,idx] = 1 e IND[grp+1,idx] = 0, SUM(IND[g,idx]) = 0 para todo g > grp
             #                                    big_M*(1 - VRGnI)  >= (VRGn+1I+VRGn+2I+...+VRGNI) - big_M*(VRGn+1I)
             # - Se VRGnI == 0 e VRGn+1I == 0   ==>   big_M*1  >= (VRGn+1I+VRGn+2I+...+VRGNI) - big_M*0
             #                                  ==>      big_M >= (VRGn+1I+VRGn+2I+...+VRGNI)

             # - Se VRGnI == 0 e VRGn+1I == 1   ==>   big_M*1 >=  (VRGn+1I+VRGn+2I+...+VRGNI) - big_M*1
             #                                  ==>     big_M >=  (VRGn+1I+VRGn+2I+...+VRGNI) - big_M

             # - Se VRGnI == 1 e VRGn+1I == 1   ==>   big_M*0 >=  (VRGn+1I+VRGn+2I+...+VRGNI) - big_M*1
             #                                  ==>         0 >=  (VRGn+1I+VRGn+2I+...+VRGNI) - big_M

             # - Se VRGnI == 1 e VRGn+1I == 0   ==>   big_M*0 >=  (VRGn+1I+VRGn+2I+...+VRGNI) - big_M*0
             #                                  ==>         0 >=  (VRGn+1I+VRGn+2I+...+VRGNI)



    # for idx in idx_list:
    #     for grp in groups_list[:-1]:
    #         big_M = n_groups - grp
    #         prob += (
    #             big_M*(1 - assignment_vars[grp,idx])  >= lpSum([assignment_vars[grp2,idx] for grp2 in groups_list[grp:]]) - big_M*(1 - assignment_vars[grp+1,idx]) ,
    #             f"PreencherGruposMenorProMaiorGrupo(2)_g{grp}_i{idx}"
    #         )# Quando IND[grp,idx] = 0 e IND[grp+1,idx] = 1, SUM(IND[g,idx]) = 0 para todo g < grp
    #          #                                      big_M*(VRGnI)  >= (VRGn-1I+VRGn-2I+...+VRG1I) - big_M*(1 - VRGn+1I)
    #          # - Se VRGnI == 0 e VRGn+1I == 0   ==>  big_M*0 >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M*1
    #          #                                  ==>        0 >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M

    #          # - Se VRGnI == 0 e VRGn+1I == 1   ==>  big_M*0 >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M*0
    #          #                                  ==>        0 >=  (VRGn-1I+VRGn-2I+...+VRG1I)

    #          # - Se VRGnI == 1 e VRGn+1I == 1   ==>  big_M*1 >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M*0
    #          #                                  ==>    big_M >=  (VRGn-1I+VRGn-2I+...+VRG1I)

    #          # - Se VRGnI == 1 e VRGn+1I == 0   ==>  big_M*1 >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M*1
    #          #                                  ==>    big_M >=  (VRGn-1I+VRGn-2I+...+VRG1I) - big_M
    

    
    return prob, assignment_vars

In [21]:
pct_per_grp = 0.1

prob, assignment_vars = generate_best_gh_problem(df, safra, perf, score, n_groups, pct_per_grp)

In [22]:
prob

BestGH:
MINIMIZE
1*desvio_medio_grp_1 + 1*desvio_medio_grp_2 + 1*desvio_medio_grp_3 + 1*desvio_medio_grp_4 + 0
SUBJECT TO
DefinicaoDesvioMedio_g1: - desvio_absoluto_ref_grp_2306_1
 - desvio_absoluto_ref_grp_2307_1 - desvio_absoluto_ref_grp_2308_1
 + 3 desvio_medio_grp_1 = 0

DefinicaoDesvioMedio_g2: - desvio_absoluto_ref_grp_2306_2
 - desvio_absoluto_ref_grp_2307_2 - desvio_absoluto_ref_grp_2308_2
 + 3 desvio_medio_grp_2 = 0

DefinicaoDesvioMedio_g3: - desvio_absoluto_ref_grp_2306_3
 - desvio_absoluto_ref_grp_2307_3 - desvio_absoluto_ref_grp_2308_3
 + 3 desvio_medio_grp_3 = 0

DefinicaoDesvioMedio_g4: - desvio_absoluto_ref_grp_2306_4
 - desvio_absoluto_ref_grp_2307_4 - desvio_absoluto_ref_grp_2308_4
 + 3 desvio_medio_grp_4 = 0

DefinicaoDesvioAbsoluto(1)_g1_r2306: bad_ref_grp_2306_1
 - desvio_absoluto_ref_grp_2306_1 - media_grp_1 <= 0

DefinicaoDesvioAbsoluto(1)_g1_r2307: bad_ref_grp_2307_1
 - desvio_absoluto_ref_grp_2307_1 - media_grp_1 <= 0

DefinicaoDesvioAbsoluto(1)_g1_r2308: bad_r

In [23]:
prob.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/thommic/Python/venvs/best_gh/lib64/python3.12/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/5c11a1b42f264fc29b5cdecdf3732fe8-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/5c11a1b42f264fc29b5cdecdf3732fe8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1047 COLUMNS
At line 13690 RHS
At line 14733 BOUNDS
At line 15711 ENDATA
Problem MODEL has 1042 rows, 993 columns and 10716 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.02 seconds
Cgl0002I 18 variables fixed
Cgl0003I 8 fixed, 0 tightened bounds, 224 strengthened rows, 1 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 1 strengthened rows, 0 substitutions
Cgl0004I processed model has 761 rows, 931 columns (903 integer (900 of which binary)) and 8391 elements
Cbc0038I Initial state - 283 integers u

1

In [24]:
for var in prob.variables():
    print(f'Var: {var} | Value: {var.value()}')

Var: bad_ref_grp_2306_1 | Value: 0.0
Var: bad_ref_grp_2306_2 | Value: 0.0
Var: bad_ref_grp_2306_3 | Value: 0.22222222
Var: bad_ref_grp_2306_4 | Value: 0.9
Var: bad_ref_grp_2307_1 | Value: 0.0
Var: bad_ref_grp_2307_2 | Value: 0.0
Var: bad_ref_grp_2307_3 | Value: 0.22222222
Var: bad_ref_grp_2307_4 | Value: 0.9
Var: bad_ref_grp_2308_1 | Value: 0.0
Var: bad_ref_grp_2308_2 | Value: 0.0
Var: bad_ref_grp_2308_3 | Value: 0.22222222
Var: bad_ref_grp_2308_4 | Value: 0.9
Var: desvio_absoluto_ref_grp_2306_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_3 | Value: 0.0
Var: desvio_absoluto_ref_grp_2306_4 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_3 | Value: 0.0
Var: desvio_absoluto_ref_grp_2307_4 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_1 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_2 | Value: 0.0
Var: desvio_absoluto_ref_grp_2308_3 | Value: 

In [26]:
refs_list = sorted([int(sfr) for sfr in df[safra].unique()])
groups_list = sorted([int(grp) for grp in np.arange(1,n_groups+1)])
idx_list = [i+1 for i in range(0,len(df[score]))]

In [27]:
solution_dict = {}

for grp in groups_list:
    solution_dict[int(grp)] = []
    
    for idx in idx_list:
        if assignment_vars[int(grp),int(idx)].value() == 1:
            solution_dict[int(grp)].append(idx)

solution_dict

{1: [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73],
 2: [74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98],
 3: [99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126],
 4: [127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
 

In [28]:
groups_assignment_list = []
for idx in idx_list:
    for grp in groups_list:
        if assignment_vars[int(grp),int(idx)].value() == 1:
            groups_assignment_list.append(grp)

df['GRUPO'] = groups_assignment_list
df

,SAFRA,PERF,SCORE,GRUPO
0,2306,0,0.000000,1
1,2308,0,0.004255,1
2,2308,0,0.008511,1
3,2306,0,0.012766,1
4,2307,0,0.017021,1
...,...,...,...,...
231,2307,1,0.982979,4
232,2306,1,0.987234,4
233,2307,1,0.991489,4
234,2308,1,0.995745,4


In [29]:
from IPython.display import display_html

#tab1 = df.reset_index(names='IDX').loc[df[df.SAFRA==refs_list[0]].index]
tabs = [df[df.SAFRA==ref]._repr_html_() for ref in refs_list]

#display_html('<div style="display: flex; flex-direction: row; justify-content: space-evenly"> '+tab1._repr_html_()+tab2._repr_html_() + '</div>', raw=True)
display_html('<div style="display: flex; flex-direction: row; justify-content: space-evenly"> '+ ''.join(tabs) + '</div>', raw=True)

,SAFRA,PERF,SCORE,GRUPO
0,2306,0,0.000000,1
3,2306,0,0.012766,1
9,2306,0,0.038298,1
11,2306,0,0.046809,1
13,2306,0,0.055319,1
...,...,...,...,...
214,2306,0,0.910638,4
223,2306,1,0.948936,4
228,2306,1,0.970213,4
230,2306,1,0.978723,4


In [32]:
tabs_ref = [df[df.SAFRA==ref] for ref in refs_list]

tabs_refs_grps = []
tabs_refs_grps_html = []
for tab in tabs_ref:
    for grp in groups_list:
        tabs_refs_grps.append(tab[tab.GRUPO==grp])
        tabs_refs_grps_html.append(tab[tab.GRUPO==grp]._repr_html_())


display_html('''<div style="display: flex; 
                            flex-direction: row; 
                            justify-content: space-evenly; 
                            flex-wrap: wrap; 
                            width: ''' + str(260*n_groups) + '''px;
                            align-content: center;
                            
                            "> '''+ ''.join(tabs_refs_grps_html) + '</div>', raw=True)
#margin-left: 150px;margin-right: 100px

,SAFRA,PERF,SCORE,GRUPO
0,2306,0,0.000000,1
3,2306,0,0.012766,1
9,2306,0,0.038298,1
11,2306,0,0.046809,1
13,2306,0,0.055319,1
16,2306,0,0.068085,1
19,2306,0,0.080851,1
21,2306,0,0.089362,1
25,2306,0,0.106383,1
29,2306,0,0.123404,1


In [33]:
abc = pd.DataFrame({"count" : df.groupby(['SAFRA','GRUPO']).PERF.count(), 
              "bad": df.groupby(['SAFRA','GRUPO']).PERF.sum(),
              "bad rate": df.groupby(['SAFRA','GRUPO']).PERF.sum()/df.groupby(['SAFRA','GRUPO']).PERF.count()
             }).unstack(0)

abc['media bad rate',''] = abc.loc[:,"bad rate"].mean(axis=1)

for ref in refs_list:
    abc.loc[:,("desvio absoluto", ref)] = (pd.Series(abc.loc[:,"media bad rate"].values.reshape(abc.shape[0],), index=abc.index) - abc.loc[:,("bad rate",ref)]).abs()

abc['media desvio padrao',''] = abc.loc[:,"desvio absoluto"].mean(axis=1)

abc.drop(columns=[("count",),("bad",)])

bad rate                     media bad rate desvio absoluto            \
SAFRA      2306      2307      2308                           2306      2307   
GRUPO                                                                          
1      0.000000  0.000000  0.000000       0.000000        0.000000  0.000000   
2      0.000000  0.000000  0.000000       0.000000        0.000000  0.000000   
3      0.200000  0.076923  0.600000       0.292308        0.092308  0.215385   
4      0.189189  0.281250  0.341463       0.270634        0.081445  0.010616   

                media desvio padrao  
SAFRA      2308                      
GRUPO                                
1      0.000000            0.000000  
2      0.000000            0.000000  
3      0.307692            0.205128  
4      0.070829            0.054297